In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os

df_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(df_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# step1: check the missing values:
print("Missing values:")
print(df.isnull().sum())

# step2: handling:
# step2.1: we separate numerical and categorical to fill the numerical missing values with the median and the categorical missing values with the mode

num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# step3: re-check
print('-----------------------')
print("Missing values after handling:")
print(df.isnull().sum())


In [ ]:
# Task 2: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

# Step: Label Encoding
label_encoder = LabelEncoder()

for col in cat_cols:
    df[col] = label_encoder.fit_transform(df[col])

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Exclude target column from scaling
X = df.drop("Target", axis=1)
y = df["Target"]

scaled_X = scaler.fit_transform(X)

# Convert back to DataFrame
X_scaled = pd.DataFrame(scaled_X, columns=X.columns)


In [ ]:
# Task 5: Write your code here:
print(y.value_counts(normalize=True))

# check imbalance
if y.value_counts(normalize=True).min() < 0.3:
    print("The target variable is imbalanced.")
else:
    print("The target variable is not imbalanced.")

# additional plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Plot target imbalance
plt.figure(figsize=(6,4))
sns.countplot(x=y, palette="viridis")

plt.title("Target Variable Distribution")
plt.xlabel("Target (0 = No Default, 1 = Default)")
plt.ylabel("Count")
plt.show()


In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df["Target"]


In [ ]:
!pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_scores = []
f1_scores = []

for train_idx, test_idx in skf.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = CatBoostClassifier(
        verbose=0,
        random_state=42
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    accuracy_scores.append(accuracy_score(y_test, preds))
    f1_scores.append(f1_score(y_test, preds))

print("Average Accuracy:", sum(accuracy_scores) / len(accuracy_scores))
print("Average F1 Score:", sum(f1_scores) / len(f1_scores))


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns

#extracting
importances = model.get_feature_importance()
feature_names = X.columns

#create a DataFrame for plotting
fi_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

#plotting
plt.figure(figsize=(10, 8))
sns.barplot(data=fi_df.head(20), x="Importance", y="Feature", palette="viridis")
plt.title("Top 20 Feature Importances (CatBoost)")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.show()

In [ ]:
# Task 2: Write your code here:
golden_feature = fi_df.iloc[0]["Feature"]
print("The Golden Feature is:", golden_feature)


In [ ]:
# Task Bonus: Write your code here:
# 1. Create X with only the golden feature
X_golden = df[[golden_feature]]

# 2. Run StratifiedKFold with CatBoost
golden_accuracy_scores = []
golden_f1_scores = []

for train_idx, test_idx in skf.split(X_golden, y):
    X_train, X_test = X_golden.iloc[train_idx], X_golden.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model_golden = CatBoostClassifier(verbose=0, random_state=42)
    model_golden.fit(X_train, y_train)

    preds = model_golden.predict(X_test)

    golden_accuracy_scores.append(accuracy_score(y_test, preds))
    golden_f1_scores.append(f1_score(y_test, preds))

# 3. Print n comparison
print("Full Model Accuracy:", sum(accuracy_scores) / len(accuracy_scores))
print("Golden Feature Accuracy:", sum(golden_accuracy_scores) / len(golden_accuracy_scores))

print("\nFull Model F1 Score:", sum(f1_scores) / len(f1_scores))
print("Golden Feature F1 Score:", sum(golden_f1_scores) / len(golden_f1_scores))
